# InDel mutation rate inside tandem repeats

InDels called inside repeat tracts by `workflows/repetitive_indel_call.py`, turned
into a rate per motif and compared between the repeat models.

**Input:** `results/repetitive_indels/{species}/{sample}/{model}/`
- `indels.tsv`: one row per read carrying an InDel whose size is a multiple of the
  period k, after the final filters (haploid concordance, kmerQ ≥ 93, at least
  100 bp from the read end). The tract it sits in is given by `canonical_motif`,
  `k` and `copies`.
- `opportunities.tsv`: read bases inside repeat tracts with kmerQ ≥ 93, counted
  per contig, motif, k, tissue and mapQ.
- `filter_report.tsv`: rows removed by each final filter.

**Rate:** for a motif, the InDels in tracts of that motif divided by the
opportunities in tracts of that motif, i.e. InDels per read base inside such
tracts. Calls the pileup flagged non-unique are dropped throughout
(`unique_only` in the setup cell); the section on that flag shows what it costs. Overlapping tracts are resolved before calling
(`resolve_repeat_overlaps.py`), so every base belongs to one tract and the
numerator and denominator agree on its motif. The haploid check and the read-end
filter remove calls but have no counterpart in the opportunities, so the rate is
a conservative estimate.

**Models:** M0 = mreps exact repeats, M1 = TRF, M2 = mreps allowing one
substitution between neighbouring copies. Each model has its own tracts, so a
model's rate covers only the sequence that model calls repetitive. Set `models`
in the next cell.

In [ ]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(yaml)

results_dir      <- "../results/repetitive_indels"
repeats_dir      <- "../results/repetitive_dna"   # tract summaries, for the rate by tract length
config_path      <- "../config.yaml"              # sample ages and groups come from here
models           <- c("M0", "M1")  # add "M2" to include it
species_keep     <- NULL           # e.g. c("human", "chimp"); NULL keeps every species
max_hap_distance <- 0            # bp; 0 keeps only calls whose haploid InDel is at the same position
unique_only      <- TRUE           # TRUE drops calls the pileup flagged non-unique, see "Unique and non-unique calls"
slippage_only    <- FALSE          # TRUE counts only InDels that are whole copies of the tract's motif
min_indels_shown <- 5              # motifs with fewer InDels in every model are left out of the motif plot
length_breaks    <- c(0, 8, 12, 16, 24, 32, 48, 64, 128, Inf)  # tract length bins (bp)

figures_dir      <- "../plots"      # where save_svg() writes

options(repr.plot.width = 11, repr.plot.height = 6)

# Save a plot for a slide or a figure. SVG keeps the text as text, so PowerPoint
# and Illustrator can restyle it, and it stays sharp at any size.
save_svg <- function(plot, file, width = 11, height = 5.5) {
  dir.create(figures_dir, showWarnings = FALSE, recursive = TRUE)
  path <- file.path(figures_dir, file)
  device <- if (requireNamespace("svglite", quietly = TRUE)) svglite::svglite else grDevices::svg
  ggsave(path, plot, width = width, height = height, device = device)
  cat("wrote", path, "\n")
  invisible(path)
}

config <- yaml::read_yaml(config_path)

# Age of the individual in years, from config.yaml, ready to join on species and
# sample. Samples with no age on the sample sheet are simply absent from it.
sample_ages <- {
  entries <- config[["sample_age"]]
  bind_rows(lapply(names(entries), function(sp)
    tibble(species = sp,
           sample  = names(entries[[sp]]),
           age     = as.numeric(unlist(entries[[sp]], use.names = FALSE)))))
}

# One fixed colour per model, the same in every notebook in this folder.
model_colours <- c(M0 = "#2a78d6", M1 = "#eb6834", M2 = "#1baf7a")
type_colours  <- c(deletion = "#52514e", insertion = "#b0aea8")

# Labels for the tract length bins: "<8", "8-11", ..., ">=128".
bin_labels <- function(breaks) {
  low  <- head(breaks, -1)
  high <- tail(breaks, -1)
  ifelse(low == 0, paste0("<", high),
         ifelse(is.infinite(high), paste0(">=", low), paste0(low, "-", high - 1)))
}
length_bin <- function(length) cut(length, length_breaks, labels = bin_labels(length_breaks), right = FALSE)

# TRUE when the file is there and has at least one row under the header. Some
# samples have no calls in a model, and read_tsv() then guesses every column as
# text, which does not stack with the other samples.
has_rows <- function(paths) {
  vapply(paths, function(p) {
    file.exists(p) && length(readLines(p, n = 2, warn = FALSE)) > 1
  }, logical(1), USE.NAMES = FALSE)
}

# The calls, with the columns that can be empty for a whole file read as text.
read_calls <- function(path) {
  read_tsv(path, show_col_types = FALSE, progress = FALSE,
           col_types = cols(tissue = col_character(), other_ops = col_character(),
                            contig_hap = col_character()))
}

# Rate with an exact 95% Poisson interval, from columns indels and opportunities.
add_rate <- function(d, level = 0.95) {
  a <- (1 - level) / 2
  mutate(d,
         rate = indels / opportunities,
         low  = qchisq(a, 2 * indels) / 2 / opportunities,
         high = qchisq(1 - a, 2 * (indels + 1)) / 2 / opportunities)
}

# A rate of 0 cannot go on a log axis, so it is drawn as an open marker at the
# top of its interval instead. Every row keeps a range (zero-length for those),
# so points and ranges dodge together.
rate_points <- function(d) {
  mutate(d,
         shown    = ifelse(indels > 0, rate, high),
         bar_low  = ifelse(indels > 0, low, high),
         estimate = ifelse(indels > 0, "InDels observed", "none, upper bound"))
}

rate_layers <- function(dodge_width = 0.6) {
  dodge <- position_dodge(width = dodge_width)
  list(
    geom_linerange(aes(ymin = bar_low, ymax = high), position = dodge, linewidth = 0.5),
    geom_point(aes(shape = estimate), position = dodge, size = 2.2),
    scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL),
    scale_y_log10()
  )
}

# Smallest rotation of a sequence or of its reverse complement, the form
# canonicalise() in scripts/parse_trf.py gives the tract motifs.
rotations <- function(s) {
  n <- nchar(s)
  substring(strrep(s, 2), seq_len(n), seq_len(n) + n - 1)
}

canonicalise <- function(seqs) {
  vapply(seqs, function(s) {
    rc <- paste(rev(strsplit(chartr("ACGT", "TGCA", s), "")[[1]]), collapse = "")
    min(rotations(s), rotations(rc))
  }, character(1), USE.NAMES = FALSE)
}

# TRUE when the InDel is whole copies of the tract's motif, in any phase and on
# either strand: CACA in an AC tract, but not AGAT.
motif_consistent <- function(indel, motif, k) {
  unit <- substr(indel, 1, k)
  strrep(unit, nchar(indel) %/% k) == indel & canonicalise(unit) == motif
}

## What is in the results folder

In [ ]:
indel_files <- list.files(results_dir, pattern = "^indels\\.tsv$",
                          recursive = TRUE, full.names = TRUE)
if (length(indel_files) == 0) stop("No indels.tsv under ", results_dir)

runs <- tibble(dir = dirname(indel_files)) |>
  mutate(
    parts   = strsplit(sub(paste0("^", results_dir, "/"), "", dir), "/"),
    species = vapply(parts, `[`, character(1), 1),
    sample  = vapply(parts, `[`, character(1), 2),
    model   = vapply(parts, `[`, character(1), 3)
  ) |>
  select(-parts) |>
  mutate(
    label = paste(species, sample, sep = " / "),
    # Not every model directory has both files: a sample can have no calls in a
    # model, and a copy from the cluster can be incomplete.
    has_calls         = has_rows(file.path(dir, "indels.tsv")),
    has_opportunities = has_rows(file.path(dir, "opportunities.tsv")),
    has_report        = has_rows(file.path(dir, "filter_report.tsv")),
    has_classified    = has_rows(file.path(dir, "indels_classified.tsv"))
  ) |>
  left_join(sample_ages, by = c("species", "sample"))

runs |>
  mutate(present = case_when(has_calls & has_opportunities ~ "both",
                             has_calls                     ~ "calls only",
                             has_opportunities             ~ "opportunities only",
                             TRUE                          ~ "empty")) |>
  select(species, sample, model, present) |>
  pivot_wider(names_from = model, values_from = present, values_fill = "-") |>
  arrange(species, sample) |>
  print(n = 40)

runs <- filter(runs, model %in% models)
if (!is.null(species_keep)) runs <- filter(runs, species %in% species_keep)
if (nrow(runs) == 0) stop("None of ", paste(models, collapse = ", "), " under ", results_dir)

# A rate needs both files, so these samples appear in the counts but in no rate.
if (any(!runs$has_opportunities)) {
  cat("\nno opportunities.tsv, so counted but not turned into a rate:\n  ",
      paste(unique(paste0(runs$label, " ", runs$model)[!runs$has_opportunities]),
            collapse = ", "), "\n")
}
if (any(!runs$has_calls)) {
  cat("\nno calls at all:\n  ",
      paste(unique(paste0(runs$label, " ", runs$model)[!runs$has_calls]), collapse = ", "), "\n")
}

In [ ]:
# Rows each final filter removed from the calls. Only the kmerQ threshold is
# applied to the opportunities as well.
runs |>
  filter(has_report) |>
  mutate(report = lapply(file.path(dir, "filter_report.tsv"), read_tsv,
                         show_col_types = FALSE, progress = FALSE)) |>
  select(species, sample, model, report) |>
  unnest(report) |>
  mutate(share_kept = round(rows_after / rows_before, 3)) |>
  print(n = 40)

In [ ]:
# The calls, with the InDel size in repeat units and the tract length in bp
# (k * copies, as in tract_lengths.ipynb).
if (!any(runs$has_calls)) stop("None of the samples has any calls")

indels_all <- file.path(runs$dir[runs$has_calls], "indels.tsv") |>
  lapply(read_calls) |>
  bind_rows()

n_calls <- nrow(indels_all)
indels_all <- indels_all |>
  filter(is.na(distance_to_indel_in_hap) | distance_to_indel_in_hap <= max_hap_distance) |>
  mutate(
    size       = nchar(indel_string),
    units      = size %/% k,
    length     = round(k * copies),
    consistent = motif_consistent(indel_string, canonical_motif, k),
    label      = paste(species, sample, sep = " / "),
    model      = factor(model, levels = models)
  ) |>
  filter(consistent | !slippage_only)

# Every rate below uses `indels`. `indels_all` keeps the non-unique calls as
# well, for the comparison in the next section.
indels <- indels_all |>
  filter(unique == "unique" | !unique_only) |>
  group_by(species, sample, model, contig, position, indel_type, indel_string) |>
  mutate(reads_at_site = n()) |>
  ungroup()

cat("InDels:", nrow(indels), "of", n_calls, "kept with max_hap_distance =", max_hap_distance,
    ", unique_only =", unique_only, ", slippage_only =", slippage_only, "\n")
cat("samples with calls:", n_distinct(indels$label), "| samples with a rate:",
    n_distinct(runs$label[runs$has_opportunities]), "\n")

# The opportunities, summed over contig and mapQ. The calls are not filtered on
# mapQ, so the denominator is not either. The model comes from the folder name,
# since the table has no model column. Tissue is kept for the rate per tissue
# below and summed over everywhere else.
if (!any(runs$has_opportunities)) stop("No opportunities.tsv under ", results_dir)

opportunities_tissue <- file.path(runs$dir[runs$has_opportunities], "opportunities.tsv") |>
  lapply(function(path) {
    read_tsv(path, show_col_types = FALSE, progress = FALSE,
             col_types = cols(tissue = col_character())) |>
      group_by(species, sample, canonical_motif, k, tissue) |>
      summarise(opportunities = sum(count), .groups = "drop") |>
      mutate(model = basename(dirname(path)))
  }) |>
  bind_rows() |>
  mutate(model = factor(model, levels = models))

opportunities <- opportunities_tissue |>
  group_by(species, sample, model, canonical_motif, k) |>
  summarise(opportunities = sum(opportunities), .groups = "drop")

# Every motif with calls should have opportunities, since both come from the
# same tracts.
no_opportunities <- indels |>
  semi_join(filter(runs, has_opportunities) |>
              distinct(species, sample, model) |>
              mutate(model = factor(model, levels = models)),
            by = c("species", "sample", "model")) |>
  count(species, sample, model, canonical_motif, k) |>
  anti_join(opportunities, by = c("species", "sample", "model", "canonical_motif", "k"))
if (nrow(no_opportunities) > 0) {
  warning(nrow(no_opportunities), " motif(s) have InDels but no opportunities")
}

cat("opportunities:", format(sum(opportunities$opportunities), big.mark = ","),
    "read bases over", nrow(distinct(opportunities, model, canonical_motif, k)), "model x motif pairs\n")

# TRF's consensus can be longer or shorter than the period it reports, so a few
# M1 motifs do not have k bases. They are kept, under the period TRF gave.
opportunities |>
  group_by(model) |>
  summarise(motifs_not_k_long = sum(nchar(canonical_motif) != k),
            share_of_opportunities = sum(opportunities[nchar(canonical_motif) != k]) / sum(opportunities)) |>
  print()

## Overview per sample and model

`sites` counts distinct InDels (same position, type and sequence), so it is lower
than `indels` when several reads carry the same one. `consistent_share` is the
share of calls that are whole copies of the tract's motif.

In [ ]:
overview <- indels |>
  group_by(species, sample, label, model) |>
  summarise(
    indels           = n(),
    sites            = n_distinct(contig, position, indel_type, indel_string),
    deletion_share   = mean(indel_type == "deletion"),
    one_unit_share   = mean(units == 1),
    consistent_share = mean(consistent),
    .groups = "drop"
  ) |>
  left_join(opportunities |>
              group_by(species, sample, model) |>
              summarise(opportunities = sum(opportunities), .groups = "drop"),
            by = c("species", "sample", "model")) |>
  mutate(rate = indels / opportunities)

print(overview, width = Inf)

## Unique and non-unique calls, k = 2

`unique` is set during pileup, before any filter: a position counts as "unique"
when at most one read there carries an InDel of any size. Inside repeats most
positions have several reads with some InDel, so most calls come out
"non-unique" even when only one read carries this particular InDel. A non-unique
call can be a real event that several reads caught, but also a variant the
assembly did not resolve, or a position where the sequencer slips again and
again.

How much the rate depends on that flag, for the dinucleotides: the same
denominator throughout, the numerator once with every call and once with the
unique ones only. This section is the only one that keeps the non-unique calls;
everywhere else `unique_only` in the setup cell decides, and it is on.

In [ ]:
k2 <- filter(indels_all, k == 2)

k2_counts <- opportunities |>
  filter(k == 2) |>
  left_join(count(k2, species, sample, model, canonical_motif, name = "n_all"),
            by = c("species", "sample", "model", "canonical_motif")) |>
  left_join(count(filter(k2, unique == "unique"), species, sample, model, canonical_motif, name = "n_unique"),
            by = c("species", "sample", "model", "canonical_motif")) |>
  mutate(across(c(n_all, n_unique), \(n) coalesce(n, 0L)),
         label = paste(species, sample, sep = " / "))

k2_counts |>
  mutate(rate_all     = n_all / opportunities,
         rate_unique  = n_unique / opportunities,
         unique_share = round(n_unique / n_all, 3),
         rate_ratio   = round(rate_unique / rate_all, 3)) |>
  select(label, model, canonical_motif, opportunities, n_all, n_unique, unique_share,
         rate_all, rate_unique, rate_ratio) |>
  arrange(label, model, canonical_motif) |>
  print(n = 60, width = Inf)

k2_rates <- k2_counts |>
  pivot_longer(c(n_all, n_unique), names_to = "calls_kept", values_to = "indels") |>
  mutate(calls_kept = recode(calls_kept, n_all = "every call", n_unique = "unique only")) |>
  add_rate()

options(repr.plot.width = 11, repr.plot.height = 3 + 2 * n_distinct(k2_rates$label))
k2_rates |>
  rate_points() |>
  ggplot(aes(x = canonical_motif, y = shown, colour = model, group = model)) +
  rate_layers() +
  scale_colour_manual(values = model_colours) +
  facet_grid(label ~ calls_kept) +
  labs(title = "Unique and non-unique calls at k = 2",
       x = "motif", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal()
options(repr.plot.width = 11, repr.plot.height = 6)

## Rate per period

All motifs of a period pooled. Intervals are exact 95% Poisson intervals on the
InDel count.

In [ ]:
rate_k <- opportunities |>
  group_by(species, sample, model, k) |>
  summarise(opportunities = sum(opportunities), .groups = "drop") |>
  left_join(count(indels, species, sample, model, k, name = "indels"),
            by = c("species", "sample", "model", "k")) |>
  mutate(indels = coalesce(indels, 0L),
         label  = paste(species, sample, sep = " / ")) |>
  add_rate()

rate_k |>
  select(label, model, k, indels, opportunities, rate, low, high) |>
  print(n = 60)

rate_k |>
  rate_points() |>
  ggplot(aes(x = factor(k), y = shown, colour = model, group = model)) +
  rate_layers() +
  scale_colour_manual(values = model_colours) +
  facet_wrap(~ label) +
  labs(title = "InDel rate by period",
       x = "period k", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal()

## Rate per motif

The rate asked for: InDels in tracts of a motif divided by the opportunities in
tracts of the same motif. Motifs are in canonical form, so a motif, its
rotations and its reverse complement share one label (AC covers CA, GT and TG).

In [ ]:
motif_rates <- opportunities |>
  left_join(count(indels, species, sample, model, canonical_motif, k, name = "indels"),
            by = c("species", "sample", "model", "canonical_motif", "k")) |>
  mutate(indels = coalesce(indels, 0L),
         label  = paste(species, sample, sep = " / ")) |>
  add_rate() |>
  arrange(label, model, desc(indels))

# The motifs carrying the most InDels in each model.
motif_rates |>
  group_by(label, model) |>
  slice_head(n = 12) |>
  ungroup() |>
  select(label, model, canonical_motif, k, indels, opportunities, rate, low, high) |>
  print(n = 80)

In [ ]:
# Motifs with at least min_indels_shown InDels in one of the models, shown in
# every model that has tracts of them.
shown_motifs <- motif_rates |>
  group_by(label, canonical_motif, k) |>
  filter(max(indels) >= min_indels_shown) |>
  ungroup()

cat(n_distinct(shown_motifs$canonical_motif), "motifs with at least",
    min_indels_shown, "InDels in one model\n")

options(repr.plot.width = 12, repr.plot.height = 5 + 2 * n_distinct(shown_motifs$label))
shown_motifs |>
  rate_points() |>
  ggplot(aes(x = reorder(canonical_motif, -rate, max), y = shown,
             colour = model, group = model)) +
  rate_layers() +
  scale_colour_manual(values = model_colours) +
  facet_grid(label ~ k, scales = "free_x", space = "free_x",
             labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "InDel rate by motif",
       x = "motif", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
options(repr.plot.width = 11, repr.plot.height = 6)

In [ ]:
# The same motif under two models, for motifs with InDels in both. Points above
# the dashed line have a higher rate in the second model.
if (length(models) < 2) {
  message("Only one model selected, nothing to compare")
} else {
  pair <- models[1:2]
  motif_pair <- motif_rates |>
    filter(model %in% pair) |>
    select(label, canonical_motif, k, model, indels, rate) |>
    pivot_wider(names_from = model, values_from = c(indels, rate), values_fill = list(indels = 0L)) |>
    filter(.data[[paste0("indels_", pair[1])]] > 0, .data[[paste0("indels_", pair[2])]] > 0) |>
    mutate(indels = .data[[paste0("indels_", pair[1])]] + .data[[paste0("indels_", pair[2])]])

  print(ggplot(motif_pair, aes(x = .data[[paste0("rate_", pair[1])]],
                               y = .data[[paste0("rate_", pair[2])]])) +
    geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey60") +
    geom_point(aes(size = indels), colour = "#52514e", alpha = 0.7) +
    geom_text(aes(label = canonical_motif), size = 3, vjust = -1, check_overlap = TRUE) +
    scale_x_log10() +
    scale_y_log10() +
    scale_size_area(max_size = 6) +
    facet_wrap(~ label) +
    labs(title = paste("Rate per motif,", pair[1], "against", pair[2]),
         x = paste(pair[1], "rate"), y = paste(pair[2], "rate"), size = "InDels") +
    theme_minimal())
}

## Rate by tract length

Tract length matters most for repeat mutation rates, and it is also where the
models differ most: M1 reports longer tracts than M0 (see `tract_lengths.ipynb`).
If M0 and M1 give similar rates at the same tract length, the difference in their
overall rates comes from which tracts they report, not from how InDels are
counted in them.

The opportunity table has no tract length, so the denominator per length bin is
approximated. Each period's opportunities are split over length bins in
proportion to the repeat sequence in each bin, taken from
`results/repetitive_dna/{species}/{sample}/{model}_summary.tsv`. This assumes
that each base of repeat sequence gets the same number of opportunities, whatever
the tract length. The check below shows how far that holds between periods and
models.

In [ ]:
summary_paths <- runs |>
  distinct(species, sample, model) |>
  mutate(path = file.path(repeats_dir, species, sample, paste0(model, "_summary.tsv")))
missing_summaries <- summary_paths$path[!file.exists(summary_paths$path)]
if (length(missing_summaries) > 0) {
  warning("No tract summary for: ", paste(missing_summaries, collapse = ", "))
}

# Repeat sequence (bp) per model, period and tract length bin.
tract_bases <- summary_paths$path[file.exists(summary_paths$path)] |>
  lapply(read_tsv, show_col_types = FALSE, progress = FALSE) |>
  bind_rows() |>
  mutate(length     = round(k * copies),
         length_bin = length_bin(length),
         model      = factor(model, levels = models)) |>
  group_by(species, sample, model, k, length_bin) |>
  summarise(bases = sum(length * count), .groups = "drop")

# Opportunities per base of repeat sequence: read bases passing kmerQ per
# reference base. Similar values across periods and models support the
# approximation.
opportunities |>
  group_by(species, sample, model, k) |>
  summarise(opportunities = sum(opportunities), .groups = "drop") |>
  left_join(tract_bases |>
              group_by(species, sample, model, k) |>
              summarise(bases = sum(bases), .groups = "drop"),
            by = c("species", "sample", "model", "k")) |>
  mutate(per_base = round(opportunities / bases, 2)) |>
  select(species, sample, model, k, per_base) |>
  pivot_wider(names_from = k, values_from = per_base, names_prefix = "k = ") |>
  print(width = Inf)

In [ ]:
length_rates <- tract_bases |>
  group_by(species, sample, model, k) |>
  mutate(share = bases / sum(bases)) |>
  ungroup() |>
  left_join(opportunities |>
              group_by(species, sample, model, k) |>
              summarise(opportunities_k = sum(opportunities), .groups = "drop"),
            by = c("species", "sample", "model", "k")) |>
  mutate(opportunities = opportunities_k * share) |>
  left_join(indels |>
              mutate(length_bin = length_bin(length)) |>
              count(species, sample, model, k, length_bin, name = "indels"),
            by = c("species", "sample", "model", "k", "length_bin")) |>
  mutate(indels = coalesce(indels, 0L),
         label  = paste(species, sample, sep = " / ")) |>
  filter(opportunities > 0) |>
  add_rate()

# Bins with too few opportunities give intervals spanning orders of magnitude.
min_opportunities_shown <- 1e5

options(repr.plot.width = 12, repr.plot.height = 4 + 2.5 * n_distinct(length_rates$label))
length_rates |>
  filter(opportunities >= min_opportunities_shown) |>
  rate_points() |>
  ggplot(aes(x = length_bin, y = shown, colour = model, group = model)) +
  rate_layers() +
  scale_colour_manual(values = model_colours) +
  facet_grid(label ~ k, labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "InDel rate by tract length",
       x = "tract length (bp)", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
options(repr.plot.width = 11, repr.plot.height = 6)

### Tract length summarised, all samples pooled

The grid above has a row per sample, which is for checking that the trend is not
one sample's. This is the same numbers pooled: calls and read bases summed
across samples, one line per model, a panel per period. One figure to show.

**The denominator here is apportioned, not counted.** The opportunity tables are
keyed by contig, motif, period, tissue, mapQ and kmerQ, with no tract length, so
each period's read bases are split across the length bins in proportion to the
repeat bases in the `repetitive_dna` tract summaries. The numerator is exact:
every call carries its tract's `copies`. So a bin's rate assumes read coverage is
flat across tract lengths within a period. Treat the levels as approximate; the
shape across bins is what the plot is for.

Pooling also mixes species, so a bin's level is a read-weighted average over
genomes of different repeat content.

In [ ]:
length_summary <- length_rates |>
  group_by(model, k, length_bin) |>
  summarise(indels = sum(indels), opportunities = sum(opportunities), .groups = "drop") |>
  filter(opportunities >= min_opportunities_shown) |>
  add_rate()

cat(sum(length_summary$indels), "calls over",
    format(sum(length_summary$opportunities), big.mark = ","),
    "apportioned read bases, in", n_distinct(length_rates$label), "samples\n")

length_summary |>
  select(model, k, length_bin, indels, opportunities, rate) |>
  mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
  print(n = 40)

dodge <- position_dodge(width = 0.6)
p_length_summary <- length_summary |>
  rate_points() |>
  ggplot(aes(x = length_bin, y = shown, colour = model, group = model)) +
  geom_line(position = dodge, linewidth = 0.4, alpha = 0.7) +
  rate_layers() +
  scale_colour_manual(values = model_colours) +
  facet_wrap(~ k, nrow = 1, labeller = labeller(k = function(x) paste("period", x))) +
  labs(title = "InDel rate by tract length",
       x = "tract length (bp)", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        panel.grid.minor = element_blank(),
        legend.position = "bottom",
        plot.title.position = "plot", plot.title = element_text(hjust = 0.5))

options(repr.plot.width = 13, repr.plot.height = 5.5)
print(p_length_summary)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_length_summary, "tract_length.svg", width = 12, height = 5.5)

## Rate per tissue

Reads carry a tissue label when the sample has a read classification (it is
listed under `tissue_samples` in config.yaml); reads without one count as NA.
Both the InDels and the opportunities carry the label, so unlike the rate by
tract length this one needs no approximation: each tissue's calls are divided by
that tissue's own read bases. Coverage differences between tissues therefore
cancel out. A sample with no labels shows NA only, and the cell says so.

In [ ]:
tissue_rates <- opportunities_tissue |>
  group_by(species, sample, model, k, tissue) |>
  summarise(opportunities = sum(opportunities), .groups = "drop") |>
  left_join(count(indels, species, sample, model, k, tissue, name = "indels"),
            by = c("species", "sample", "model", "k", "tissue")) |>
  mutate(indels = coalesce(indels, 0L),
         tissue = ifelse(is.na(tissue), "NA (unlabelled)", tissue),
         label  = paste(species, sample, sep = " / "))

# Over all periods, and then per period.
tissue_totals <- tissue_rates |>
  group_by(label, model, tissue) |>
  summarise(indels = sum(indels), opportunities = sum(opportunities), .groups = "drop") |>
  add_rate()
print(tissue_totals, n = 40)

# Only samples whose reads carry labels go in the plots; the others would add
# a row of empty tissues.
labelled_samples <- tissue_totals |>
  filter(tissue != "NA (unlabelled)") |>
  group_by(label) |>
  filter(n_distinct(tissue) >= 2) |>
  pull(label) |>
  unique()

if (length(labelled_samples) == 0) {
  message("No sample has two or more tissue labels, nothing to compare. Only samples listed ",
          "under tissue_samples in config.yaml have read labels.")
} else {
  print(tissue_totals |>
    filter(label %in% labelled_samples) |>
    rate_points() |>
    ggplot(aes(x = tissue, y = shown, colour = model, group = model)) +
    rate_layers() +
    scale_colour_manual(values = model_colours) +
    facet_wrap(~ label, scales = "free_x") +
    labs(title = "InDel rate by tissue, all periods",
         x = NULL, y = "InDels per read base in tracts", colour = NULL) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)))

  options(repr.plot.width = 12, repr.plot.height = 4 + 2.5 * length(labelled_samples))
  print(tissue_rates |>
    filter(label %in% labelled_samples) |>
    add_rate() |>
    rate_points() |>
    ggplot(aes(x = tissue, y = shown, colour = model, group = model)) +
    rate_layers() +
    scale_colour_manual(values = model_colours) +
    facet_grid(label ~ k, labeller = labeller(k = function(x) paste0("k = ", x))) +
    labs(title = "InDel rate by tissue and period",
         x = NULL, y = "InDels per read base in tracts", colour = NULL) +
    theme_minimal() +
    theme(axis.text.x = element_text(angle = 45, hjust = 1)))
  options(repr.plot.width = 11, repr.plot.height = 6)
}

### Rate per tissue and period, read ends removed

The counterpart of the tissue plot in `nonrepetitive_indels.ipynb`: only the
samples listed under `tissue_samples` in config.yaml, only unique calls, and
only calls further than 300 bp from a read end, where the near-end excess has
died away. Periods 2, 3 and 4, one panel each, one row of panels per model.

The denominator is the read bases inside tracts of that period, summed over
motifs, for that tissue. Cutting the numerator at 300 bp means cutting it too:
a read of length L holds `min(2 * 300, L)` positions within 300 bp of one of its
ends, so each tissue's read bases are scaled by the share outside. That share is
estimated per sample and tissue from the reads carrying a call there, which are
slightly longer than average, so the correction is a small underestimate — and
near enough equal across the tissues of one sample that the comparison between
them is unaffected.

One thing to keep in mind at k = 4: the caller reports InDels of 1 to 3 bp, so
nothing in a 4 bp tract can be a whole copy of the motif. Those panels count
partial-copy events only, which is a different kind of event from the one-copy
slippage that dominates k = 2 and k = 3.

In [ ]:
read_end_cut   <- 300         # calls at or below this distance from a read end are dropped
tissue_periods <- c(2, 3, 4)  # periods to show

# Which samples carry read labels, from config.yaml, as in nonrepetitive_indels.ipynb.
tissue_keys <- {
  entries <- yaml::read_yaml("../config.yaml")[["tissue_samples"]]
  unlist(lapply(names(entries), function(sp) paste(sp, entries[[sp]], sep = "/")),
         use.names = FALSE)
}

tissue_runs <- runs |>
  filter(has_opportunities, paste(species, sample, sep = "/") %in% tissue_keys) |>
  distinct(species, sample)

if (nrow(tissue_runs) == 0) {
  message("No sample under tissue_samples in config.yaml has opportunities here.")
} else {
  # indels_all rather than indels, so the cell does not depend on unique_only.
  tissue_calls <- indels_all |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    filter(unique == "unique", ToreadEnd > read_end_cut, !is.na(tissue),
           k %in% tissue_periods)

  # The share of a tissue's read bases left after the cut. A read of length L
  # holds min(2 * cut, L) positions within `cut` of an end.
  usable <- indels_all |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    filter(!is.na(tissue)) |>
    distinct(species, sample, tissue, readID, read_length) |>
    group_by(species, sample, tissue) |>
    summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
              reads = n(), .groups = "drop")

  tissue_rates_trim <- opportunities_tissue |>
    semi_join(tissue_runs, by = c("species", "sample")) |>
    filter(!is.na(tissue), k %in% tissue_periods) |>
    group_by(species, sample, model, k, tissue) |>
    summarise(positions = sum(opportunities), .groups = "drop") |>
    inner_join(usable, by = c("species", "sample", "tissue")) |>
    left_join(count(tissue_calls, species, sample, model, k, tissue, name = "indels"),
              by = c("species", "sample", "model", "k", "tissue")) |>
    mutate(indels        = coalesce(indels, 0L),
           opportunities = positions * usable_share,
           label         = paste(species, sample, sep = " / ")) |>
    add_rate()

  cat(nrow(tissue_calls), "unique calls beyond", read_end_cut, "bp from a read end, in",
      n_distinct(tissue_rates_trim$label), "samples,",
      n_distinct(tissue_rates_trim$tissue), "tissues and periods",
      paste(tissue_periods, collapse = ", "), "\n")
  cat("read bases kept by the cut:",
      paste0(round(100 * range(tissue_rates_trim$usable_share), 1), "%", collapse = " to "), "\n")

  tissue_rates_trim |>
    select(label, tissue, model, k, indels, opportunities, rate) |>
    mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
    arrange(label, tissue, model, k) |>
    print(n = 40)

  # Tissues ordered by their rate across samples, so the ranking is the y axis.
  dodge <- position_dodge(width = 0.7)
  p_tissue_rate <- tissue_rates_trim |>
    rate_points() |>
    mutate(tissue = reorder(tissue, rate, median)) |>
    ggplot(aes(x = shown, y = tissue, colour = species)) +
    geom_linerange(aes(xmin = bar_low, xmax = high, group = label),
                   position = dodge, linewidth = 0.4) +
    geom_point(aes(shape = estimate, group = label), position = dodge, size = 1.9) +
    scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL) +
    scale_x_log10() +
    facet_grid(model ~ k, labeller = labeller(k = function(x) paste("period", x))) +
    labs(title = "InDel rate by tissue",
         x = "rate", y = NULL, colour = NULL) +
    theme_minimal(base_size = 13) +
    theme(legend.position = "bottom",
    plot.title    = element_text(hjust = 0.5),
          panel.grid.minor = element_blank(),
          plot.title.position = "plot")

  options(repr.plot.width = 13, repr.plot.height = 4 + 2.6 * n_distinct(tissue_rates_trim$model))
  print(p_tissue_rate)
  options(repr.plot.width = 11, repr.plot.height = 6)

  save_svg(p_tissue_rate, "tissue_rate_repeats.svg", width = 12, height = 7)
}

### Sperm against age

The same comparison as in `nonrepetitive_indels.ipynb`, inside the tracts. Two
sources of sperm reads, kept apart by the marker: the samples under
`sperm_samples` in config.yaml, where the whole sample is sperm, and the
sperm-labelled reads inside the multi-tissue samples, where the denominator is
that tissue's own read bases.

Unique calls further than 300 bp from a read end, periods 2, 3 and 4, with the
denominator scaled by the share of read bases surviving the cut. Samples with no
unique calls at all are dropped rather than drawn as a rate of zero.

The line is a fit through the human sperm series alone, the only within-species
age range here; the other points are single animals of different species. It is
a quasi-Poisson regression of the counts with the tract read bases as offset, so
the coefficient is a rate ratio per year of age.

In [ ]:
sperm_keys <- {
  entries <- config[["sperm_samples"]]
  unlist(lapply(names(entries), function(sp) paste(sp, entries[[sp]], sep = "/")),
         use.names = FALSE)
}

sperm_source <- runs |>
  filter(has_opportunities) |>
  distinct(species, sample, label, age) |>
  mutate(key       = paste(species, sample, sep = "/"),
         is_sperm  = key %in% sperm_keys,
         is_tissue = key %in% tissue_keys) |>
  filter(is_sperm | is_tissue) |>
  mutate(from = ifelse(is_sperm, "whole sample", "sperm reads")) |>
  select(species, sample, label, from, age)

# A whole-sample sperm run has no tissue column to filter on; a multi-tissue
# sample contributes only the reads labelled sperm.
is_sperm_row <- function(d) filter(d, from == "whole sample" | tissue == "sperm")

sperm_calls <- indels_all |>
  inner_join(sperm_source, by = c("species", "sample")) |>
  is_sperm_row() |>
  filter(unique == "unique", ToreadEnd > read_end_cut, k %in% tissue_periods)

usable <- indels_all |>
  inner_join(sperm_source, by = c("species", "sample")) |>
  is_sperm_row() |>
  distinct(species, sample, readID, read_length) |>
  group_by(species, sample) |>
  summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
            .groups = "drop")

sperm_rates <- opportunities_tissue |>
  inner_join(sperm_source, by = c("species", "sample")) |>
  is_sperm_row() |>
  filter(k %in% tissue_periods) |>
  group_by(species, sample, label, from, age, model, k) |>
  summarise(positions = sum(opportunities), .groups = "drop") |>
  left_join(usable, by = c("species", "sample")) |>
  left_join(count(sperm_calls, species, sample, model, k, name = "indels"),
            by = c("species", "sample", "model", "k")) |>
  mutate(indels        = coalesce(indels, 0L),
         usable_share  = coalesce(usable_share, median(usable_share, na.rm = TRUE)),
         opportunities = positions * usable_share) |>
  add_rate()

no_calls <- sperm_rates |>
  group_by(label) |>
  filter(sum(indels) == 0) |>
  pull(label) |>
  unique()
if (length(no_calls) > 0) {
  cat("dropped, no unique calls at all:", paste(no_calls, collapse = ", "), "\n")
  sperm_rates <- filter(sperm_rates, !label %in% no_calls)
}

cat(nrow(sperm_calls), "unique sperm calls beyond", read_end_cut, "bp from a read end, in",
    n_distinct(sperm_rates$label), "samples\n")
if (any(is.na(sperm_rates$age))) {
  cat("no age, so not in the plot:",
      paste(unique(sperm_rates$label[is.na(sperm_rates$age)]), collapse = ", "), "\n")
}

sperm_rates |>
  select(label, from, age, model, k, indels, opportunities, rate) |>
  mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
  arrange(from, label, model, k) |>
  print(n = 60)

# The age trend within the human sperm series, per model and period.
age_fit <- sperm_rates |>
  filter(from == "whole sample", !is.na(age), indels > 0) |>
  group_by(model, k) |>
  group_modify(~ {
    if (nrow(.x) < 3) return(tibble())
    m <- glm(indels ~ age + offset(log(opportunities)), family = quasipoisson, data = .x)
    s <- summary(m)$coefficients
    tibble(samples  = nrow(.x),
           per_year = round(exp(s["age", 1]), 4),
           ci_low   = round(exp(s["age", 1] - 1.96 * s["age", 2]), 4),
           ci_high  = round(exp(s["age", 1] + 1.96 * s["age", 2]), 4),
           p        = signif(s["age", 4], 2))
  }) |>
  ungroup()
cat("\nrate ratio per year of age, human sperm samples only:\n")
print(as.data.frame(age_fit), row.names = FALSE)

p_sperm_age <- sperm_rates |>
  filter(!is.na(age)) |>
  rate_points() |>
  ggplot(aes(x = age, y = shown)) +
  geom_smooth(data = ~ filter(.x, from == "whole sample", indels > 0),
              method = "lm", formula = y ~ x,
              colour = "grey30", fill = "grey80", linewidth = 0.5, alpha = 0.3) +
  geom_linerange(aes(ymin = bar_low, ymax = high, colour = species), linewidth = 0.4) +
  geom_point(aes(colour = species, shape = from, alpha = estimate), size = 2.2) +
  scale_alpha_manual(values = c("InDels observed" = 1, "none, upper bound" = 0.45), name = NULL) +
  ggrepel::geom_text_repel(aes(label = sample), size = 2.7, colour = "grey25",
                           min.segment.length = 0.2, seed = 1, max.overlaps = 20) +
  scale_shape_manual(values = c("whole sample" = 16, "sperm reads" = 17)) +
  scale_y_log10() +
  facet_grid(model ~ k, labeller = labeller(k = function(x) paste("period", x))) +
  labs(title = "Sperm InDel rate against age, in tracts",
       x = "age of the animal (years)", y = "InDels per read base in tracts",
       colour = NULL, shape = NULL) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank(),
        plot.title.position = "plot", plot.title = element_text(hjust = 0.5))

options(repr.plot.width = 13, repr.plot.height = 4 + 2.6 * n_distinct(sperm_rates$model))
print(p_sperm_age)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_sperm_age, "sperm_age_repeats.svg", width = 12, height = 7)

### The human sperm samples pooled, by motif

The counterpart of the pooled context spectrum in `nonrepetitive_indels.ipynb`.
All human samples under `sperm_samples` in config.yaml, their calls and their
tract read bases summed into one spectrum per motif. `human/la4` has no calls at
all and `human/tsaed` none that are unique, so both are dropped rather than
contributing a denominator with an empty numerator.

Unique calls further than 300 bp from a read end, periods 2, 3 and 4, with each
sample's tract read bases scaled by the share surviving that cut before they are
summed. Deletions and insertions share one denominator per motif, since an
opportunity is a read base inside a tract of that motif, where the tract can
contract or expand.

A motif whose pooled tract bases fall below `min_positions_shown` is left out:
with a handful of tracts the interval spans orders of magnitude and tells you
only that the motif is rare.

In [ ]:
min_positions_shown <- 1e6   # pooled tract read bases a motif needs to be shown

pool_runs <- runs |>
  filter(species == "human", has_calls, has_opportunities,
         paste(species, sample, sep = "/") %in% sperm_keys) |>
  distinct(species, sample)

pool_calls <- indels_all |>
  semi_join(pool_runs, by = c("species", "sample")) |>
  filter(unique == "unique", ToreadEnd > read_end_cut, k %in% tissue_periods)

# A sample contributing a denominator but no unique calls would drag the pooled
# rate down for a reason that is not biological.
no_calls <- setdiff(pool_runs$sample, unique(pool_calls$sample))
if (length(no_calls) > 0) {
  cat("dropped, no unique calls at all:", paste(no_calls, collapse = ", "), "\n")
  pool_runs <- filter(pool_runs, !sample %in% no_calls)
  pool_calls <- filter(pool_calls, !sample %in% no_calls)
}
cat("pooled:", paste(sort(pool_runs$sample), collapse = ", "), "\n")

usable <- indels_all |>
  semi_join(pool_runs, by = c("species", "sample")) |>
  distinct(species, sample, readID, read_length) |>
  group_by(species, sample) |>
  summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
            .groups = "drop")

pool_rates <- opportunities |>
  semi_join(pool_runs, by = c("species", "sample")) |>
  filter(k %in% tissue_periods) |>
  inner_join(usable, by = c("species", "sample")) |>
  mutate(positions = opportunities * usable_share) |>
  group_by(model, k, canonical_motif) |>
  summarise(opportunities = sum(positions), .groups = "drop") |>
  cross_join(tibble(indel_type = c("deletion", "insertion"))) |>
  left_join(count(pool_calls, model, k, canonical_motif, indel_type, name = "indels"),
            by = c("model", "k", "canonical_motif", "indel_type")) |>
  mutate(indels = coalesce(indels, 0L)) |>
  add_rate()

shown <- filter(pool_rates, opportunities >= min_positions_shown)
cat(sum(shown$indels), "unique calls over",
    format(sum(distinct(shown, model, k, canonical_motif, opportunities)$opportunities),
           big.mark = ","), "tract read bases;",
    n_distinct(paste(pool_rates$model, pool_rates$k, pool_rates$canonical_motif)) -
      n_distinct(paste(shown$model, shown$k, shown$canonical_motif)),
    "motif-model pairs below the floor are not shown\n")

shown |>
  select(model, k, canonical_motif, indel_type, indels, opportunities, rate) |>
  mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
  arrange(desc(rate)) |>
  print(n = 20)

p_sperm_motif <- shown |>
  rate_points() |>
  ggplot(aes(x = reorder(canonical_motif, -rate, max), y = shown,
             colour = indel_type, group = indel_type)) +
  rate_layers() +
  scale_colour_manual(values = type_colours) +
  facet_grid(model ~ k, scales = "free_x", space = "free_x",
             labeller = labeller(k = function(x) paste("period", x))) +
  labs(title = "Human sperm pooled, InDel rate by motif",
       x = "motif", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
        panel.grid.minor = element_blank(),
        legend.position = "bottom",
        plot.title.position = "plot", plot.title = element_text(hjust = 0.5))

options(repr.plot.width = 13, repr.plot.height = 4 + 2.6 * n_distinct(shown$model))
print(p_sperm_motif)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_sperm_motif, "sperm_motif_repeats.svg", width = 12, height = 7)

### The same motifs, split by tract length

The pooled sperm spectrum again, with each motif's tracts divided into short and
long. The split is at **48 bp**, not 100: at 48 bp the calls divide almost
evenly (48% long, 52% short) and the largest number of motifs keep enough calls
on both sides to compare, while a 100 bp split leaves only a quarter of the
calls on the long side. `length_split` changes it.

**How the denominator is split.** The opportunity tables carry no tract length,
so each motif's read bases are divided between short and long in proportion to
that motif's reference tract bases, taken from the `repetitive_dna` summaries,
which do carry `copies` per motif. That is one level finer than the apportioning
in the tract-length plot above, where only the period was used, but it rests on
the same assumption: that read coverage does not depend on tract length within a
motif. The numerator is exact, since every call carries its tract's `copies`.

Long tracts are a small share of the sequence — 8.5% of tract bases at this
split — so the long side has wider intervals despite holding half the calls.
Deletions and insertions are pooled here, which leaves the colour for the model.

In [ ]:
length_split <- 48   # bp; tracts at or above this are "long"

# Reference tract bases per motif and length class, from the repeat summaries.
# The summaries carry their own species, sample and model columns.
motif_bases <- pool_runs |>
  cross_join(tibble(model_name = models)) |>
  mutate(path = file.path(repeats_dir, species, sample, paste0(model_name, "_summary.tsv"))) |>
  filter(file.exists(path)) |>
  pull(path) |>
  lapply(read_tsv, show_col_types = FALSE, progress = FALSE) |>
  bind_rows() |>
  filter(k %in% tissue_periods) |>
  mutate(length  = round(k * copies),
         tract   = ifelse(length >= length_split,
                          paste0("long tracts (>= ", length_split, " bp)"),
                          paste0("short tracts (< ", length_split, " bp)")),
         model   = factor(model, levels = models)) |>
  group_by(species, sample, model, k, canonical_motif, tract) |>
  summarise(bases = sum(length * count), .groups = "drop")

# Each motif's read bases divided between the two classes in proportion to its
# reference tract bases.
motif_positions <- motif_bases |>
  group_by(species, sample, model, k, canonical_motif) |>
  mutate(share = bases / sum(bases)) |>
  ungroup() |>
  inner_join(opportunities, by = c("species", "sample", "model", "k", "canonical_motif")) |>
  inner_join(usable, by = c("species", "sample")) |>
  mutate(positions = opportunities * share * usable_share) |>
  group_by(model, k, canonical_motif, tract) |>
  summarise(opportunities = sum(positions), .groups = "drop")

split_rates <- motif_positions |>
  left_join(pool_calls |>
              mutate(length = round(k * copies),
                     tract  = ifelse(length >= length_split,
                                     paste0("long tracts (>= ", length_split, " bp)"),
                                     paste0("short tracts (< ", length_split, " bp)"))) |>
              count(model, k, canonical_motif, tract, name = "indels"),
            by = c("model", "k", "canonical_motif", "tract")) |>
  mutate(indels = coalesce(indels, 0L),
         tract  = factor(tract, levels = c(paste0("short tracts (< ", length_split, " bp)"),
                                           paste0("long tracts (>= ", length_split, " bp)")))) |>
  add_rate()

shown_split <- split_rates |>
  filter(opportunities >= min_positions_shown) |>
  group_by(model, k, canonical_motif) |>
  filter(n_distinct(tract) == 2) |>    # keep motifs that survive on both sides
  ungroup()

cat(sum(shown_split$indels), "calls over",
    format(sum(shown_split$opportunities), big.mark = ","), "read bases;",
    n_distinct(paste(shown_split$model, shown_split$k, shown_split$canonical_motif)),
    "motifs shown on both sides of", length_split, "bp\n")

shown_split |>
  select(model, k, canonical_motif, tract, indels, opportunities, rate) |>
  mutate(opportunities = signif(opportunities, 3), rate = signif(rate, 3)) |>
  arrange(model, k, canonical_motif, tract) |>
  print(n = 30)

# How much longer tracts raise the rate, motif by motif.
shown_split |>
  mutate(class = ifelse(grepl("^long", tract), "long", "short")) |>
  select(model, k, canonical_motif, class, rate) |>
  pivot_wider(names_from = class, values_from = rate) |>
  mutate(long_over_short = round(long / short, 1)) |>
  arrange(desc(long_over_short)) |>
  print(n = 30)

# Short and long side by side for each motif, one model per row.
tract_colours <- setNames(c("#898781", "#2a78d6"), levels(shown_split$tract))

p_motif_split <- shown_split |>
  rate_points() |>
  ggplot(aes(x = reorder(canonical_motif, -rate, max), y = shown,
             colour = tract, group = tract)) +
  rate_layers() +
  scale_colour_manual(values = tract_colours) +
  facet_grid(model ~ k, scales = "free_x", space = "free_x",
             labeller = labeller(k = function(x) paste("period", x))) +
  labs(title = "Human sperm pooled, InDel rate by motif and tract length",
       x = "motif", y = "InDels per read base in tracts", colour = NULL) +
  theme_minimal(base_size = 13) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
        panel.grid.minor = element_blank(),
        legend.position = "bottom",
        strip.text.y = element_text(angle = 0, hjust = 0),
        plot.title.position = "plot", plot.title = element_text(hjust = 0.5))

options(repr.plot.width = 13, repr.plot.height = 4 + 2.6 * n_distinct(shown_split$model))
print(p_motif_split)
options(repr.plot.width = 11, repr.plot.height = 6)

save_svg(p_motif_split, "sperm_motif_by_length.svg", width = 12, height = 7.5)

## What the InDels look like

Size in repeat units, signed: negative for deletions (a tract losing copies),
positive for insertions (gaining copies). Shares are within a model and period.

In [ ]:
change_labels <- c("<=-5", -4:-1, 1:4, ">=5")

spectrum <- indels |>
  mutate(change = ifelse(indel_type == "insertion", units, -units),
         change = factor(pmax(pmin(change, 5), -5), levels = c(-5:-1, 1:5),
                         labels = change_labels)) |>
  count(label, model, k, change) |>
  group_by(label, model, k) |>
  mutate(share = n / sum(n)) |>
  ungroup()

options(repr.plot.width = 12, repr.plot.height = 4 + 2.5 * n_distinct(spectrum$label))
ggplot(spectrum, aes(x = change, y = share, fill = model)) +
  geom_col(position = position_dodge(width = 0.8, preserve = "single"), width = 0.75) +
  scale_fill_manual(values = model_colours) +
  facet_grid(label ~ k, labeller = labeller(k = function(x) paste0("k = ", x))) +
  labs(title = "InDel size in repeat units",
       x = "change in copies", y = "share of InDels", fill = NULL) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
options(repr.plot.width = 11, repr.plot.height = 6)

### Calls that are not copies of the motif

Slippage adds or removes whole copies of the motif. In an exact (M0) tract, any
deletion that lies entirely inside the tract is copies of the motif in some
phase, so a deletion that is not must reach past the tract. The caller assigns
an InDel to the tract containing the base before it, so an InDel starting just
after a tract's last base is assigned to that tract. Insertions can carry any
sequence, so an insertion that is not copies of the motif is not slippage.

The reverse also matters: if the aligner shifts InDels in a repeat to their
leftmost equivalent position, a copy gained or lost in an exact tract is
reported at the base before the tract, which is outside it, and is not called.
Impurities in M1 tracts stop that shift inside the tract. A low consistent share
in M0 next to a high one in M1 fits this. `slippage_only` in the setup cell
restricts every rate in this notebook to calls that are copies of the motif.

In [ ]:
# Per period: deletions against insertions, and how many calls are whole copies
# of the tract's motif, for deletions and insertions separately.
indels |>
  group_by(label, model, k) |>
  summarise(indels                   = n(),
            deletion_share           = round(mean(indel_type == "deletion"), 3),
            one_unit_share           = round(mean(units == 1), 3),
            consistent_share         = round(mean(consistent), 3),
            consistent_in_deletions  = round(mean(consistent[indel_type == "deletion"]), 3),
            consistent_in_insertions = round(mean(consistent[indel_type == "insertion"]), 3),
            .groups = "drop") |>
  print(n = 60, width = Inf)

# The calls that do not match their tract's motif, most common first.
indels |>
  filter(!consistent) |>
  count(label, model, canonical_motif, indel_string, sort = TRUE) |>
  group_by(label, model) |>
  slice_head(n = 8) |>
  print(n = 40)

## Recurrent sites

`reads_at_site` counts the reads that carry the same InDel at the same position,
among the calls this notebook analyses. A new mutation should be on one read;
the same InDel on several is more likely a variant the assembly did not resolve.

The `unique` flag is a different thing, set during pileup before any filter:
"unique" when at most one read at that position has an InDel of any size. The
second table shows how the calls split on it, before `unique_only` is applied.

In [ ]:
# Distinct InDels by how many analysed reads carry them.
indels |>
  distinct(label, model, contig, position, indel_type, indel_string, reads_at_site) |>
  mutate(reads = ifelse(reads_at_site >= 5, "5+", as.character(reads_at_site))) |>
  count(label, model, reads) |>
  pivot_wider(names_from = reads, values_from = n, values_fill = 0) |>
  print()

# The pileup flag, before unique_only is applied.
indels_all |>
  count(label, model, unique) |>
  group_by(label, model) |>
  mutate(share = round(n / sum(n), 3)) |>
  ungroup() |>
  print()

## Haploid check

`haploid_check` is TRUE when, in the read's alignment to the haploid assembly,
the first InDel of the same type at or after the call's read position has the
same size. How far away that InDel is (`distance_to_indel_in_hap`) is not
limited, so a call can pass on an InDel several kb away. `max_hap_distance` in
the setup cell requires the two to be close. The table shows how much a given
threshold would remove (the calls shown are those left after the current
`max_hap_distance`).

In [ ]:
indels |>
  mutate(distance = cut(distance_to_indel_in_hap, c(-Inf, 0, 10, 100, 1000, Inf),
                        labels = c("0", "1-10", "11-100", "101-1000", ">1000"))) |>
  count(label, model, distance) |>
  group_by(label, model) |>
  mutate(share = round(n / sum(n), 3)) |>
  ungroup() |>
  select(-n) |>
  pivot_wider(names_from = distance, values_from = share, values_fill = 0) |>
  print(width = Inf)

ggplot(indels, aes(x = distance_to_indel_in_hap + 1, colour = model)) +
  stat_ecdf(linewidth = 0.6) +
  scale_x_log10() +
  scale_colour_manual(values = model_colours) +
  facet_wrap(~ label) +
  labs(title = "Distance to the haploid InDel",
       x = "distance + 1 (bp)", y = "share of calls at most this far", colour = NULL) +
  theme_minimal()

## Germline or somatic reads

For the samples listed under `methylation_samples` in config.yaml the pipeline
writes `indels_classified.tsv` next to `indels.tsv`: the same calls, plus five
columns describing the read each call sits on.

| column | meaning |
|---|---|
| `max_germline` | best likelihood among the germline (`Sperm.*`) cell types |
| `max_somatic` | best likelihood among the somatic cell types |
| `classification_diff` | `max_germline - max_somatic`; positive looks germline |
| `classification_bin` | that difference cut into bins |
| `CpGs_onRead` | CpGs on the read, i.e. the evidence behind the likelihoods |

A difference of -1 or more counts as germline, as in the classification notebook
these columns come from.

These are counts and shares, not rates: a rate per bin needs how many reads of
the whole sample fall in each bin, which lives in the sample's
`{sample}_classification_thresholds.tsv` and is not fetched by the pipeline.

In [ ]:
classified_paths <- file.path(runs$dir[runs$has_classified], "indels_classified.tsv")

classified <- NULL
if (length(classified_paths) == 0) {
  message("No indels_classified.tsv under ", results_dir,
          ". Only samples listed under methylation_samples in config.yaml get one, ",
          "and the classification step has to have run.")
} else {
  classification <- classified_paths |>
    lapply(read_calls) |>
    bind_rows() |>
    select(species, sample, model, readID, position, indel_string,
           max_germline, max_somatic, classification_diff, classification_bin, CpGs_onRead)

  # The filters the rest of the notebook applies are already in `indels`, so the
  # columns are joined onto it rather than read again.
  # A read that is not in the sample's likelihood table has no scores, so
  # classification_diff is NA there and `germline` is neither TRUE nor FALSE.
  classified <- indels |>
    inner_join(classification,
               by = c("species", "sample", "model", "readID", "position", "indel_string")) |>
    mutate(germline = classification_diff >= -1,
           scored   = !is.na(classification_diff),
           classification_bin = factor(
             ifelse(scored, as.character(classification_bin), "not scored"),
             levels = c(as.character(sort(unique(classification_bin))), "not scored")))

  cat(nrow(classified), "of", nrow(indels), "calls joined to a read, in",
      n_distinct(classified$label), "sample(s);", sum(!classified$scored),
      "sit on a read that is not in the likelihood table\n")

  classified |>
    group_by(label, model) |>
    summarise(calls          = n(),
              # summarise() evaluates in order, so `somatic` and the share use
              # the counts just computed, not the logical columns they came from.
              not_scored     = sum(!scored),
              scored         = calls - not_scored,
              germline       = sum(germline, na.rm = TRUE),
              somatic        = scored - germline,
              share_germline = round(germline / scored, 3),
              median_diff    = round(median(classification_diff, na.rm = TRUE), 2),
              .groups = "drop") |>
    print(n = 40)
}

In [ ]:
if (!is.null(classified)) {
  options(repr.plot.width = 11, repr.plot.height = 3 + 2 * n_distinct(classified$label))
  bin_counts <- count(classified, label, model, classification_bin)
  print(ggplot(bin_counts, aes(x = classification_bin, y = n, fill = model)) +
    geom_col(position = "dodge") +
    scale_fill_manual(values = model_colours, drop = FALSE) +
    facet_wrap(~ label, ncol = 1, scales = "free_y") +
    labs(title = "Calls by classification bin",
         x = "classification_bin", y = "calls", fill = NULL) +
    theme_minimal())
  options(repr.plot.width = 11, repr.plot.height = 6)
}

### Rate per bin, on an approximate denominator

The counterpart of the same section in `nonrepetitive_indels.ipynb`. The calls
carry a bin but the opportunity tables do not: they count read bases in tracts
with read identity aggregated away, so

    P_b = the read bases in tracts contributed by the reads in bin b

is not in them. `classification_reads.tsv`, written by
`workflows/classification_denominator.py`, holds how many of the sample's reads
fall in each bin and how many CpGs they carry, over the whole likelihood table.
Each bin then takes a share of the tract read bases P of that model and period:

    P_b  ~=  P * reads_b / reads_total          (flat)
    P_b  ~=  P * CpGs_b  / CpGs_total           (length-weighted)

The counts are a property of the read, not of a pipeline or a model, so the file
is the same in both result trees and is read from whichever has it.

**What the flat split assumes.** That a read in one bin contributes as many
tract bases as a read in another. A longer read carries more CpGs, is classified
more confidently and lands in a more extreme bin, so the extreme bins hold
longer reads and the flat split gives them too small a denominator. The
length-weighted split uses CpGs as a stand-in for read length; both are
reported and the gap between them is the size of the effect.

Same filters as the tissue plot: unique calls, further than 300 bp from a read
end, periods 2, 3 and 4, with the denominator scaled per bin by the share of
that bin's read bases surviving the cut. Calls on reads absent from the
likelihood table have no bin and no denominator, so they are left out.

In [ ]:
bin_rates <- NULL

if (is.null(classified)) {
  message("No classification loaded, nothing to put a denominator under.")
} else {
  # The read counts are a property of the sample, so the file is the same in
  # both result trees; take whichever is there.
  counts_path <- function(species, sample) {
    candidates <- file.path(c(results_dir, "../results/nonrepetitive_indels"),
                            species, sample, "classification_reads.tsv")
    found <- candidates[file.exists(candidates)]
    if (length(found) == 0) NA_character_ else found[1]
  }

  read_counts <- runs |>
    distinct(species, sample, label) |>
    mutate(path = mapply(counts_path, species, sample)) |>
    filter(!is.na(path)) |>
    mutate(counts = lapply(path, read_tsv, show_col_types = FALSE, progress = FALSE)) |>
    select(species, sample, label, counts) |>
    unnest(counts) |>
    group_by(species, sample) |>
    mutate(read_share = reads / sum(reads),
           cpg_share  = CpGs / sum(CpGs),
           germline   = classification_bin >= -1) |>
    ungroup()

  if (nrow(read_counts) == 0) {
    message("No classification_reads.tsv found. Run ",
            "gwf -f workflows/classification_denominator.py run")
  } else {
    cat(n_distinct(read_counts$label), "sample(s) with read counts,",
        format(sum(read_counts$reads), big.mark = ","), "reads in all\n")

    read_counts |>
      group_by(label) |>
      summarise(reads              = sum(reads),
                share_germline     = round(sum(read_share[germline]), 3),
                cpg_share_germline = round(sum(cpg_share[germline]), 3),
                .groups = "drop") |>
      print(n = 20)

    # The calls: unique, clear of the read ends, and the periods the tissue plot
    # uses.
    scored_calls <- classified |>
      filter(scored, unique == "unique", ToreadEnd > read_end_cut,
             k %in% tissue_periods) |>
      mutate(classification_bin = as.numeric(as.character(classification_bin)))

    # The read-end cut removes tract bases from each bin too, and extreme bins
    # hold longer reads, so the share lost is estimated per bin.
    usable_bin <- scored_calls |>
      distinct(species, sample, classification_bin, readID, read_length) |>
      group_by(species, sample, classification_bin) |>
      summarise(usable_share = 1 - sum(pmin(2 * read_end_cut, read_length)) / sum(read_length),
                .groups = "drop")

    bin_rates <- opportunities |>
      semi_join(read_counts, by = c("species", "sample")) |>
      filter(k %in% tissue_periods) |>
      group_by(species, sample, model, k) |>
      summarise(positions = sum(opportunities), .groups = "drop") |>
      inner_join(select(read_counts, species, sample, label, classification_bin,
                        germline, read_share, cpg_share),
                 by = c("species", "sample"), relationship = "many-to-many") |>
      left_join(usable_bin, by = c("species", "sample", "classification_bin")) |>
      left_join(count(scored_calls, species, sample, model, k, classification_bin,
                      name = "indels"),
                by = c("species", "sample", "model", "k", "classification_bin")) |>
      mutate(indels        = coalesce(indels, 0L),
             usable_share  = coalesce(usable_share, 1),
             opportunities = positions * read_share * usable_share,
             cpg_weighted  = positions * cpg_share  * usable_share) |>
      add_rate() |>
      mutate(rate_cpg = indels / cpg_weighted)

    cat(nrow(scored_calls), "unique scored calls beyond", read_end_cut,
        "bp from a read end, over", n_distinct(bin_rates$label), "samples and periods",
        paste(tissue_periods, collapse = ", "), "\n")

    # Germline against somatic, the bins pooled either side of -1.
    germline_rates <- bin_rates |>
      group_by(label, model, k, germline) |>
      summarise(indels = sum(indels), opportunities = sum(opportunities),
                cpg_weighted = sum(cpg_weighted), .groups = "drop") |>
      add_rate() |>
      mutate(rate_cpg = indels / cpg_weighted,
             cells    = ifelse(germline, "germline", "somatic"))

    germline_rates |>
      select(label, model, k, cells, indels, rate, rate_cpg) |>
      pivot_wider(names_from = cells, values_from = c(indels, rate, rate_cpg)) |>
      mutate(ratio     = round(rate_germline / rate_somatic, 2),
             ratio_cpg = round(rate_cpg_germline / rate_cpg_somatic, 2)) |>
      select(label, model, k, indels_germline, indels_somatic, ratio, ratio_cpg) |>
      print(n = 40)

    p_classification_rate <- bin_rates |>
      rate_points() |>
      ggplot(aes(x = classification_bin, y = shown, colour = species, group = label)) +
      geom_vline(xintercept = -1.5, colour = "grey60", linetype = "dashed") +
      geom_line(linewidth = 0.4, alpha = 0.8) +
      geom_point(aes(shape = estimate), size = 1.5) +
      scale_shape_manual(values = c("InDels observed" = 16, "none, upper bound" = 1), name = NULL) +
      scale_y_log10() +
      scale_x_continuous(breaks = sort(unique(bin_rates$classification_bin))) +
      facet_grid(model ~ k, labeller = labeller(k = function(x) paste("period", x))) +
      labs(title = "InDel rate by classification bin",
           x = "classification_bin      (negative: somatic, positive: germline)",
           y = "InDels per read base in tracts", colour = NULL) +
      theme_minimal(base_size = 13) +
      theme(legend.position = "bottom",
            panel.grid.minor = element_blank(),
            plot.title.position = "plot")

    options(repr.plot.width = 13, repr.plot.height = 4 + 2.6 * n_distinct(bin_rates$model))
    print(p_classification_rate)
    options(repr.plot.width = 11, repr.plot.height = 6)

    save_svg(p_classification_rate, "classification_rate_repeats.svg", width = 12, height = 7)
  }
}

In [ ]:
if (!is.null(classified)) {
  # Only the scored reads have a difference to plot, and a read with 0 CpGs
  # cannot go on a log axis.
  shown <- filter(classified, scored, CpGs_onRead > 0)
  print(ggplot(shown, aes(x = CpGs_onRead, y = classification_diff, colour = label)) +
    geom_point(alpha = 0.3, size = 0.7) +
    geom_hline(yintercept = -1, linetype = "dashed", colour = "grey40") +
    scale_x_log10() +
    labs(title = "Evidence behind the classification",
         x = "CpGs on the read (log scale)", y = "classification_diff", colour = NULL) +
    theme_minimal())
}

### Classification against the tissue label

Two independent read labels: the tissue from `tissue_samples` and the
germline/somatic likelihoods from `methylation_samples`. A sample needs to be in
both lists for this to show anything.

In [ ]:
if (!is.null(classified)) {
  both <- filter(classified, !is.na(tissue))
  if (nrow(both) == 0) {
    message("No sample has both a tissue label and a read classification: ",
            "tissue_samples and methylation_samples in config.yaml do not overlap.")
  } else {
    crosstab <- both |>
      filter(scored) |>
      count(label, tissue, germline) |>
      pivot_wider(names_from = germline, values_from = n, values_fill = 0,
                  names_prefix = "germline_")
    print(crosstab, n = 40)

    print(ggplot(both, aes(x = tissue, y = classification_diff)) +
      geom_hline(yintercept = -1, linetype = "dashed", colour = "grey40") +
      geom_boxplot(outlier.size = 0.4) +
      facet_wrap(~ label, scales = "free_x") +
      labs(title = "Classification against tissue",
           x = NULL, y = "classification_diff") +
      theme_minimal() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1)))
  }
}